# 🔥 Colab GPU ile Derin Öğrenme Pipeline

Bu notebook, Colab GPU ortamında PyTorch ile derin öğrenme projeleri için eksiksiz ve doğru bir iskelet sunar. Her adımda açıklamalar ve kod örnekleriyle, hem eğitim hem de değerlendirme süreçlerini kolayca uygulayabilirsiniz.

## 1️⃣ Colab GPU ve Ortam Kurulumu

Colab'da GPU kullanmak için: Menüden Runtime → Change runtime type → GPU seçin. Aşağıdaki kod ile GPU erişimini ve modelinizi hangi cihazda çalıştıracağınızı kontrol edin.

In [ ]:
# GPU kontrolü ve cihaz seçimi
import torch

gpu_available = torch.cuda.is_available()
device = torch.device('cuda' if gpu_available else 'cpu')
print(f"GPU kullanılabilir mi? {gpu_available}")
if gpu_available:
    print(f"Kullanılan GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU bulunamadı, CPU kullanılacak.")

## 2️⃣ Bağımlılıkların Yüklenmesi ve Sürüm Doğrulama

Gerekli kütüphaneleri yükleyin ve sürümlerini kontrol edin. Colab'da çoğu paket önceden yüklüdür, eksikse pip ile kurabilirsiniz.

In [ ]:
# Gerekli paketlerin kurulumu (gerekirse)
!pip install torch torchvision numpy matplotlib tqdm --quiet

# Kütüphaneleri içe aktar ve sürümleri yazdır
import torch
import torchvision
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"torch: {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"numpy: {np.__version__}")
print(f"matplotlib: {matplotlib.__version__}")

## 3️⃣ Deterministik Çalışma ve Tohumlama

Deneylerin tekrarlanabilir olması için tüm rastgelelik kaynaklarını sabitleyin. Bu, model sonuçlarının her çalıştırmada aynı olmasını sağlar (bazı GPU işlemlerinde tam deterministiklik garanti edilemez).

In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Tüm seedler {SEED} olarak ayarlandı.")

## 4️⃣ Veri Üretimi ve Dataloader Hazırlığı

MNIST örneğiyle veri indirme, train/val ayrımı, DataLoader ayarları. Kendi verinizi kullanacaksanız bu bölümü özelleştirin.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Dönüşümler ve veri seti
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# MNIST veri setini indir
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Train/val ayrımı
val_size = 5000
train_size = len(train_dataset) - val_size
train_ds, val_ds = random_split(train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

batch_size = 128
num_workers = 2
pin_memory = True if torch.cuda.is_available() else False

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_dataset)}")

## 5️⃣ Model Tanımı (PyTorch)

Basit bir CNN modeli örneği. Kendi veri tipinize göre giriş/çıkış boyutlarını güncelleyebilirsiniz.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.fc1 = nn.Linear(9216, 128)
        self.dropout2 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
print(model)

## 6️⃣ Kayıp Fonksiyonu ve Optimizasyon

Sınıflandırma için CrossEntropyLoss, optimizasyon için Adam ve öğrenme oranı scheduler örneği.

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(criterion)
print(optimizer)
print(scheduler)

## 7️⃣ Eğitim Döngüsü (GPU ile)

Eğitim ve validasyon döngüsü, mixed precision (autocast) ve tqdm ile ilerleme çubuğu içerir. Her epoch sonunda doğruluk ve kayıp loglanır.

In [ ]:
from torch.cuda.amp import autocast, GradScaler

epochs = 10
scaler = GradScaler()
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        with autocast():
            output = model(data)
            loss = criterion(output, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        train_correct += (pred == target).sum().item()
        train_total += data.size(0)
    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            with autocast():
                output = model(data)
                loss = criterion(output, target)
            val_loss += loss.item() * data.size(0)
            pred = output.argmax(dim=1)
            val_correct += (pred == target).sum().item()
            val_total += data.size(0)
    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Acc={train_acc:.4f} | Val Loss={val_loss:.4f}, Acc={val_acc:.4f}")
    scheduler.step()
    # En iyi modeli kaydet
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")
        print("Yeni en iyi model kaydedildi.")

## 8️⃣ Değerlendirme ve Metrikler

Test seti üzerinde modelin doğruluğunu ve kaybını hesaplayın. İsteğe bağlı olarak confusion matrix de eklenebilir.

In [ ]:
# En iyi modeli yükle ve test et
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()
test_loss, test_correct, test_total = 0, 0, 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        test_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        test_correct += (pred == target).sum().item()
        test_total += data.size(0)
test_loss /= test_total
test_acc = test_correct / test_total
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

## 9️⃣ Model Kaydetme/Yükleme

Modelin state_dict'ini kaydetme ve yükleme örneği. Eğitim sırasında en iyi model otomatik kaydedildi, burada manuel kaydetme/yükleme de gösterilmiştir.

In [ ]:
# Modeli kaydetme
save_path = "my_model.pt"
torch.save(model.state_dict(), save_path)
print(f"Model kaydedildi: {save_path}")

# Modeli yükleme
model2 = SimpleCNN().to(device)
model2.load_state_dict(torch.load(save_path, map_location=device))
model2.eval()
print("Model başarıyla yüklendi ve eval modunda.")

## 🔟 İnferans ve Örnek Çıktılar

Rastgele bir test batch'i üzerinde modelin tahminlerini ve softmax olasılıklarını görselleştirin.

In [ ]:
import torch.nn.functional as F

# Rastgele bir batch al
model.eval()
data_iter = iter(test_loader)
data, target = next(data_iter)
data, target = data.to(device), target.to(device)
with torch.no_grad():
    output = model(data)
    probs = F.softmax(output, dim=1)
    pred = output.argmax(dim=1)

# İlk 8 örneği görselleştir
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    axes[i].imshow(data[i].cpu().squeeze(), cmap='gray')
    axes[i].set_title(f"Gerçek: {target[i].item()}\nTahmin: {pred[i].item()}")
    axes[i].axis('off')
plt.show()

print("Softmax olasılıkları (ilk örnek):", probs[0].cpu().numpy())

## 1️⃣1️⃣ Profiling ve Hız Karşılaştırması (CPU vs GPU)

Aynı batch'i CPU ve GPU'da çalıştırıp süre farkını ölçün. Büyük batch size ile GPU avantajı daha belirgin olur.

In [ ]:
import time

# Aynı batch'i CPU ve GPU'da çalıştır
batch = data.cpu()
model_cpu = SimpleCNN().cpu()
model_cpu.load_state_dict(model.state_dict())

# CPU timing
start = time.perf_counter()
with torch.no_grad():
    for _ in range(20):
        out = model_cpu(batch)
end = time.perf_counter()
print(f"CPU süresi (20 tekrar): {end-start:.3f} sn")

# GPU timing
if torch.cuda.is_available():
    batch_gpu = data.to('cuda')
    model_gpu = SimpleCNN().to('cuda')
    model_gpu.load_state_dict(model.state_dict())
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(20):
            out = model_gpu(batch_gpu)
    torch.cuda.synchronize()
    end = time.perf_counter()
    print(f"GPU süresi (20 tekrar): {end-start:.3f} sn")
else:
    print("GPU bulunamadı, sadece CPU test edildi.")